In [3]:
import matplotlib.pyplot as plt
import numpy as np

from ugdatalab import SDSSData


# 1

## Explain how our cut on log g effectively distinguishes between dwarfs and giants. Suppose you have a solar-mass star. Calculate the expected value of log g on the main sequence (when $R\sim1R_⊙$), just before the helium flash (when $R\sim100R_⊙$), and during core helium burning (when $R\sim15R_⊙$). Note that astronomers do all their calculations in cgs units.

## Visualize their distribution in label space using a corner plot.


In [4]:
sdss = SDSSData()
catalog = sdss.data
feh_col = "FE_H" if "FE_H" in catalog.colnames else "M_H"

print(f"rows in full allStar catalog: {len(catalog)}")
print(f"using metallicity column: {feh_col}")
print()
print("available label columns:")
for name in ["TEFF", "LOGG", feh_col, "MG_FE", "SI_FE"]:
    print(f"  - {name}")


InconsistentTableError: Number of header columns (1) inconsistent with data columns in data line 2

In [ ]:
label_columns = {
    "TEFF": "Teff",
    "LOGG": "log g",
    feh_col: "[Fe/H]",
    "MG_FE": "[Mg/Fe]",
    "SI_FE": "[Si/Fe]",
}


def to_float_array(column):
    values = np.ma.asarray(column, dtype=float)
    return np.asarray(np.ma.filled(values, np.nan), dtype=float)


print(f"{'column':<8} {'label':<8} {'finite':>8} {'sentinel':>10} {'valid':>8} {'min':>12} {'max':>12}")
print("-" * 74)

for name, label in label_columns.items():
    values = to_float_array(catalog[name])
    finite = np.isfinite(values)
    sentinel = finite & (values <= -9990)
    valid = finite & ~sentinel
    min_value = np.nan if not np.any(valid) else values[valid].min()
    max_value = np.nan if not np.any(valid) else values[valid].max()
    print(
        f"{name:<8} {label:<8} {finite.sum():8d} {sentinel.sum():10d} {valid.sum():8d} "
        f"{min_value:12.3f} {max_value:12.3f}"
    )


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
axes = axes.ravel()

for ax, (name, label) in zip(axes, label_columns.items()):
    values = to_float_array(catalog[name])
    valid = np.isfinite(values) & (values > -9990)
    ax.hist(values[valid], bins=60, color="C0", alpha=0.85)
    ax.set_xlabel(label)
    ax.set_ylabel("count")
    ax.set_title(name)

axes[-1].axis("off")
plt.show()
